In [8]:
!pip install snscrape pandas

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.0 MB 2.8 MB/s eta 0:00:02
   ---------- ----------------------------- 1.0/4.0 MB 3.0 MB/s eta 0:00:02
   ------------------ --------------------- 1.8/4.0 MB 3.2 MB/s eta 0:00:01
   -------------------------- ------------- 2.6/4.0 MB 3.4 MB/s eta 0:00:01
   --------------------------------- ------ 3.4/4.0 MB 3.3 MB/s eta 0:00:01
   ---------------------------------------  3.9/4.0 MB 3.3 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 3.1 MB/s  0:00:01

   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   -------------------- ------------------- 1/2 [snscrape]
   -------------------- ------------------- 1/2 [snscrape]
   -----


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import snscrape.modules.twitter as sntwitter
import pandas as pd

def scrape_tweets(keyword, label, max_tweets=1000):
    tweets = []
    
    query = f"{keyword} lang:id"
    
    for i, tweet in enumerate(sntwitter.TwitterSearchScraper(query).get_items()):
        if i >= max_tweets:
            break
        tweets.append([tweet.content, label])
    
    return tweets

data = []

# keyword per emosi
emotion_keywords = {
    "love": ["aku sayang", "cinta banget", "kangen"],
    "anger": ["marah", "kesel banget", "benci"],
    "sadness": ["sedih", "kecewa", "nangis"],
    "joy": ["senang", "bahagia", "gembira"],
    "fear": ["takut", "khawatir", "cemas"]
}

for label, keywords in emotion_keywords.items():
    for kw in keywords:
        data.extend(scrape_tweets(kw, label, max_tweets=300))

df = pd.DataFrame(data, columns=["text", "label"])
df.to_csv("dataset_emosi.csv", index=False)

print("Total data:", len(df))

Error retrieving https://twitter.com/search?f=live&lang=en&q=aku+sayang+lang%3Aid&src=spelling_expansion_revert_click: SSLError(MaxRetryError("HTTPSConnectionPool(host='twitter.com', port=443): Max retries exceeded with url: /search?f=live&lang=en&q=aku+sayang+lang%3Aid&src=spelling_expansion_revert_click (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1007)')))"))
4 requests to https://twitter.com/search?f=live&lang=en&q=aku+sayang+lang%3Aid&src=spelling_expansion_revert_click failed, giving up.
Errors: SSLError(MaxRetryError("HTTPSConnectionPool(host='twitter.com', port=443): Max retries exceeded with url: /search?f=live&lang=en&q=aku+sayang+lang%3Aid&src=spelling_expansion_revert_click (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1007)')))")), SSLError(MaxRetryError("

ScraperException: 4 requests to https://twitter.com/search?f=live&lang=en&q=aku+sayang+lang%3Aid&src=spelling_expansion_revert_click failed, giving up.

In [5]:
import asyncio
import pandas as pd
from twscrape import API

async def scrape_emotion_class(api, emotion_name, search_query, limit=1000):
    print(f"\nMulai scraping untuk kelas: {emotion_name.upper()}")
    print(f"Query: {search_query}")
    
    tweets_data = []
    
    # Menarik data menggunakan twscrape
    try:
        async for tweet in api.search(search_query, limit=limit):
            tweets_data.append({
                "id": tweet.id,
                "date": tweet.date,
                "user": tweet.user.username,
                "text": tweet.rawContent,
                "emotion_class": emotion_name # Langsung beri label untuk dataset
            })
            
            # Print progress tiap 100 data agar lo tahu script tidak nyangkut
            if len(tweets_data) % 100 == 0:
                print(f"[{emotion_name}] Terkumpul: {len(tweets_data)} / {limit}")
                
    except Exception as e:
         print(f"Error saat scraping {emotion_name}: {e}")

    # Simpan per kelas ke CSV agar jika gagal di tengah jalan, data sebelumnya aman
    if tweets_data:
        df = pd.DataFrame(tweets_data)
        filename = f"dataset_{emotion_name}.csv"
        df.to_csv(filename, index=False)
        print(f"Selesai! Data {emotion_name} disimpan ke {filename}")
    
    return tweets_data

async def main():
    api = API()
    
    # Kamus query yang spesifik agar data lo nggak penuh sampah
    queries = {
        "anger": '"gue marah banget" OR "emosi banget" OR "bikin emosi" lang:id',
        "joy": '"seneng banget" OR "akhirnya kesampaian" OR "alhamdulillah" lang:id',
        "sadness": '"nangis banget" OR "nyesek" OR "sedih banget" lang:id',
        "fear": '"takut banget" OR "merinding" OR "parno" lang:id',
        "love": '"sayang banget" OR "cinta banget" OR "gemes" lang:id'
    }

    all_data = []

    # Looping untuk kelima kelas emosi
    for emotion, query in queries.items():
        data = await scrape_emotion_class(api, emotion, query, limit=1000)
        all_data.extend(data)
        
        # Jeda 10 detik antar pencarian kelas untuk menurunkan risiko limit Twitter
        print("Jeda 10 detik sebelum lanjut ke emosi berikutnya...")
        await asyncio.sleep(10)

    # Gabungkan semua data jadi satu dataset final (5000 baris)
    if all_data:
        df_final = pd.DataFrame(all_data)
        df_final.to_csv("dataset_emosi_final.csv", index=False)
        print("\nSELURUH PROSES SELESAI. File 'dataset_emosi_final.csv' siap di-preprocess.")

await main()

2026-04-22 21:56:24.079 | WARNING  | twscrape.accounts_pool:get_for_queue_or_wait:297 - No active accounts. Stopping...
2026-04-22 21:56:24.093 | WARNING  | twscrape.accounts_pool:get_for_queue_or_wait:297 - No active accounts. Stopping...



Mulai scraping untuk kelas: ANGER
Query: "gue marah banget" OR "emosi banget" OR "bikin emosi" lang:id
Jeda 10 detik sebelum lanjut ke emosi berikutnya...


2026-04-22 21:56:34.122 | WARNING  | twscrape.accounts_pool:get_for_queue_or_wait:297 - No active accounts. Stopping...
2026-04-22 21:56:34.140 | WARNING  | twscrape.accounts_pool:get_for_queue_or_wait:297 - No active accounts. Stopping...



Mulai scraping untuk kelas: JOY
Query: "seneng banget" OR "akhirnya kesampaian" OR "alhamdulillah" lang:id
Jeda 10 detik sebelum lanjut ke emosi berikutnya...


2026-04-22 21:56:44.162 | WARNING  | twscrape.accounts_pool:get_for_queue_or_wait:297 - No active accounts. Stopping...
2026-04-22 21:56:44.179 | WARNING  | twscrape.accounts_pool:get_for_queue_or_wait:297 - No active accounts. Stopping...



Mulai scraping untuk kelas: SADNESS
Query: "nangis banget" OR "nyesek" OR "sedih banget" lang:id
Jeda 10 detik sebelum lanjut ke emosi berikutnya...


2026-04-22 21:56:54.207 | WARNING  | twscrape.accounts_pool:get_for_queue_or_wait:297 - No active accounts. Stopping...
2026-04-22 21:56:54.242 | WARNING  | twscrape.accounts_pool:get_for_queue_or_wait:297 - No active accounts. Stopping...



Mulai scraping untuk kelas: FEAR
Query: "takut banget" OR "merinding" OR "parno" lang:id
Jeda 10 detik sebelum lanjut ke emosi berikutnya...


2026-04-22 21:57:04.273 | WARNING  | twscrape.accounts_pool:get_for_queue_or_wait:297 - No active accounts. Stopping...
2026-04-22 21:57:04.296 | WARNING  | twscrape.accounts_pool:get_for_queue_or_wait:297 - No active accounts. Stopping...



Mulai scraping untuk kelas: LOVE
Query: "sayang banget" OR "cinta banget" OR "gemes" lang:id
Jeda 10 detik sebelum lanjut ke emosi berikutnya...
